# Phân tích hội tụ trên Simglucose (Type 1 Diabetes Simulator)
Notebook chuyên dụng vẽ đồ thị so sánh hiệu năng của các thuật toán Risk-Aware Bandit trên môi trường y tế thực tế **Simglucose**:
- **Robust NeuralLCB**
- **RiskExact NeuralLCB**
- **Quantile Risk (DDE)**
- **RiskLin LCB**
- **Pessimistic CDF Bandit** (Context-aware, với IS / WIS / DR estimators)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os
import glob
import re
from collections import defaultdict

# ─────────────────────────────────────────────
# 1. CẤU HÌNH (CONFIGURATION)
# ─────────────────────────────────────────────
results_root = "results"

# Các experiment muốn plot:
# None = plot tất cả (ví dụ: 'linear', 'cosine', 'simglucose')
# Hoặc lọc cụ thể: ['simglucose'] hoặc ['linear', 'simglucose']
EXP_TO_PLOT = ['simglucose']

# Các alpha muốn plot — để None thì tự động lấy tất cả các alpha có trong dữ liệu
ALPHAS_TO_PLOT = None   # ví dụ lọc: [0.05, 0.1, 0.2]

# Các thuật toán muốn plot — để None thì plot tất cả các thuật toán có trong dữ liệu
# Hoặc lọc cụ thể, ví dụ:
# ['robust', 'risk_exact', 'pessimistic_cdf_ctx_DR', 'pessimistic_cdf_ctx_IS', 'pessimistic_cdf_ctx_WIS']
ALGOS_TO_PLOT = None

# Giá trị N muốn plot — để None thì tự động lấy toàn bộ N có sẵn theo từng dataset
# Hoặc lọc cụ thể, ví dụ: [100, 500, 1000, 5000, 10000, 20000, 50000]
N_LIST = None

# Bảng màu cho từng thuật toán
color_map = {
    'robust':                   '#ff7f0e',   # Cam (Robust NeuralLCB)
    'quantile_risk':            '#1f77b4',   # Xanh dương (Direct Distributional DDE)
    'risk_lin_lcb':             '#2ca02c',   # Xanh lá (RiskLin LCB)
    'risk_exact':               '#d62728',   # Đỏ (RiskExact NeuralLCB)
    'neural_regression':        '#8c564b',   # Nâu (Neural Regression Baseline)
    
    # Pessimistic CDF - Global (arXiv:2605.15620)
    'pessimistic_cdf':          '#9467bd',   # Tím
    'pessimistic_cdf_IS':       '#8c564b',   # Nâu đỏ
    'pessimistic_cdf_WIS':      '#e377c2',   # Hồng
    'pessimistic_cdf_DR':       '#17becf',   # Cyan
    
    # Pessimistic CDF - Contextual (arXiv:2605.15620)
    'pessimistic_cdf_ctx':      '#9467bd',   # Tím
    'pessimistic_cdf_ctx_IS':   '#bcbd22',   # Vàng chanh
    'pessimistic_cdf_ctx_WIS':  '#7f7f7f',   # Xám đậm
    'pessimistic_cdf_ctx_DR':   '#9467bd',   # Tím đậm
}

# Nhãn hiển thị cho từng thuật toán
label_map = {
    'robust':                   'Robust NeuralLCB',
    'quantile_risk':            'Quantile Risk (DDE)',
    'risk_lin_lcb':             'RiskLin LCB',
    'risk_exact':               'RiskExact NeuralLCB',
    'neural_regression':        'Neural Regression',
    
    # Pessimistic CDF Global
    'pessimistic_cdf':          'Pessimistic CDF (Global)',
    'pessimistic_cdf_IS':       'Pessimistic CDF (IS)',
    'pessimistic_cdf_WIS':      'Pessimistic CDF (WIS)',
    'pessimistic_cdf_DR':       'Pessimistic CDF (DR)',
    
    # Pessimistic CDF Contextual
    'pessimistic_cdf_ctx':      'Pessimistic CDF (Contextual)',
    'pessimistic_cdf_ctx_IS':   'Pessimistic CDF-Ctx (IS)',
    'pessimistic_cdf_ctx_WIS':  'Pessimistic CDF-Ctx (WIS)',
    'pessimistic_cdf_ctx_DR':   'Pessimistic CDF-Ctx (DR)',
}

# Marker style
marker_map = {
    'robust':                   'o',
    'quantile_risk':            's',
    'risk_lin_lcb':             '^',
    'risk_exact':               'D',
    'neural_regression':        'v',
    
    'pessimistic_cdf':          'P',
    'pessimistic_cdf_IS':       'v',
    'pessimistic_cdf_WIS':      '<',
    'pessimistic_cdf_DR':       '>',
    
    'pessimistic_cdf_ctx':      '*',
    'pessimistic_cdf_ctx_IS':   'X',
    'pessimistic_cdf_ctx_WIS':  'd',
    'pessimistic_cdf_ctx_DR':   '*',
}

DEFAULT_PALETTE = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#a65628', '#f781bf', '#999999']

def get_algo_color(algo, idx=0):
    return color_map.get(algo, DEFAULT_PALETTE[idx % len(DEFAULT_PALETTE)])

def get_algo_label(algo):
    return label_map.get(algo, algo)

def get_algo_marker(algo):
    return marker_map.get(algo, 'o')

print('Cấu hình hoàn tất.')

In [ ]:
# ─────────────────────────────────────────────
# 2. HÀM TIỆN ÍCH
# ─────────────────────────────────────────────

def setup_neurips_style():
    plt.rcParams.update({
        "text.usetex": False,
        "font.family": "serif",
        "font.serif": ["DejaVu Serif", "Times New Roman"],
        "axes.labelsize": 13,
        "font.size": 13,
        "legend.fontsize": 9,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "axes.titlesize": 14,
        "lines.linewidth": 2.2,
        "lines.markersize": 7,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.dpi": 100
    })

setup_neurips_style()


def get_algo_prefix(fname):
    """
    Trích xuất prefix thuật toán từ tên file (kèm estimator IS/WIS/DR nếu có).
    Ví dụ:
      - 'robust_simglucose_...' -> 'robust'
      - 'pessimistic_cdf_ctx_simglucose_...est=dr_...' -> 'pessimistic_cdf_ctx_DR'
      - 'pessimistic_cdf_robust_syn_...est=is_...' -> 'pessimistic_cdf_IS'
    """
    m = re.match(r'^(.*?)_(robust_syn|mushroom|simglucose)_', fname)
    if not m:
        return None
    base = m.group(1)
    est_m = re.search(r'est=([a-zA-Z]+)', fname)
    if est_m:
        est = est_m.group(1).upper()
        return f"{base}_{est}"
    return base


def get_alpha_from_fname(fname):
    """Trích xuất giá trị alpha từ tên file."""
    m = re.search(r'alpha=([0-9.]+)', fname)
    return float(m.group(1)) if m else None


def get_n_from_fname(fname):
    """Trích xuất N từ tên file."""
    m = re.search(r'_n=(\d+)[_.]', fname)
    return int(m.group(1)) if m else None


def load_one_file(fpath):
    """Load một file npz và trả về dict metrics."""
    d = np.load(fpath, allow_pickle=True)
    algo_name = str(d['algo_names'][0]).replace(' ', '_') if 'algo_names' in d else None

    def get(suffix):
        if algo_name and f"{algo_name}_{suffix}" in d:
            return d[f"{algo_name}_{suffix}"]
        return d[suffix] if suffix in d else None

    out = {}
    regrets = get('regrets')
    if regrets is not None:
        out['regret_mean'] = float(np.mean(regrets))
        out['regret_sem']  = float(np.std(regrets) / np.sqrt(len(regrets.flatten())))

    gt_cvars = get('gt_cvars')
    if gt_cvars is not None:
        oracle_cvar = float(np.mean(d['oracle_cvars'])) if 'oracle_cvars' in d else 0.0
        subopts = get('subopt')
        if subopts is not None:
            subopt = subopts
        else:
            subopt = oracle_cvar - gt_cvars
        out['gt_cvar_mean'] = float(np.mean(gt_cvars))
        out['gt_cvar_sem']  = float(np.std(gt_cvars) / np.sqrt(len(gt_cvars.flatten())))
        out['subopt_mean']  = float(np.mean(subopt))
        out['subopt_sem']   = float(np.std(subopt) / np.sqrt(len(subopt.flatten())))
        out['oracle_cvar']  = oracle_cvar

    accs = get('accs')
    if accs is not None:
        out['acc_mean'] = float(np.mean(accs))
        out['acc_sem']  = float(np.std(accs) / np.sqrt(len(accs.flatten())))

    return out


def find_experiment_dirs(root_dir):
    """
    Tự động quét và tìm tất cả thư mục chứa file .npz kết quả thực nghiệm.
    Phân loại:
      - simglucose (kể cả trường hợp bị lưu trong folder std=N/A)
      - synthetic linear / cosine / ...
      - các realworld dataset khác
    """
    exp_dirs = {}
    for root, dirs, files in os.walk(root_dir):
        if os.path.abspath(root) == os.path.abspath(root_dir):
            continue
        npz_files = [f for f in files if f.endswith('.npz') and not f.endswith(':Zone.Identifier')]
        if npz_files:
            if 'simglucose' in root:
                exp_name = 'simglucose'
            elif 'robust_syn' in root:
                m = re.search(r'robust_syn_([a-zA-Z0-9]+)', root)
                exp_name = m.group(1) if m else os.path.basename(root)
            else:
                exp_name = os.path.basename(root)
            exp_dirs[exp_name] = root
    return exp_dirs


def load_all_data(data_dir, n_filter=None, alpha_filter=None, algo_filter=None):
    """
    Nạp dữ liệu từ một thư mục kết quả.
    Trả về: dict  [alpha][prefix][n] -> metrics
    """
    all_files = [f for f in glob.glob(os.path.join(data_dir, '*.npz'))
                 if not f.endswith(':Zone.Identifier')]

    result = defaultdict(lambda: defaultdict(dict))  # [alpha][prefix][n]

    for fpath in all_files:
        fname = os.path.basename(fpath)
        prefix = get_algo_prefix(fname)
        alpha  = get_alpha_from_fname(fname)
        n      = get_n_from_fname(fname)
        if prefix is None or alpha is None or n is None:
            continue
        if alpha_filter is not None and alpha not in alpha_filter:
            continue
        if algo_filter is not None and prefix not in algo_filter:
            continue
        if n_filter is not None and n not in n_filter:
            continue
        metrics = load_one_file(fpath)
        if metrics:
            result[alpha][prefix][n] = metrics

    return result

print('Hàm tiện ích đã định nghĩa.')

In [ ]:
# ─────────────────────────────────────────────
# 3. NẠP DỮ LIỆU TỰ ĐỘNG
# ─────────────────────────────────────────────

alpha_filter = set(ALPHAS_TO_PLOT) if ALPHAS_TO_PLOT else None
algo_filter  = set(ALGOS_TO_PLOT) if ALGOS_TO_PLOT else None
n_filter     = set(N_LIST) if N_LIST else None

exp_dirs = find_experiment_dirs(results_root)
all_experiment_data = {}   # exp_name -> {alpha -> {prefix -> {n -> metrics}}}

print(f"Tìm thấy các thư mục thực nghiệm: {list(exp_dirs.keys())}\n")

for exp_name in sorted(exp_dirs.keys()):
    dpath = exp_dirs[exp_name]
    if EXP_TO_PLOT is not None and exp_name not in EXP_TO_PLOT:
        continue
    print(f"Đang nạp: {exp_name} ({dpath})")
    data = load_all_data(dpath, n_filter=n_filter, alpha_filter=alpha_filter, algo_filter=algo_filter)
    if data:
        all_experiment_data[exp_name] = data

print("\n" + "="*65)
print("TỔNG KẾT DỮ LIỆU ĐÃ NẠP THÀNH CÔNG:")
print("="*65)
for exp_name, d in all_experiment_data.items():
    alphas = sorted(d.keys())
    print(f"[{exp_name}] Các alpha = {alphas}")
    for alpha in alphas:
        prefixes = sorted(d[alpha].keys())
        ns = sorted(set(n for p in prefixes for n in d[alpha][p].keys()))
        print(f"    alpha={alpha:<5}: {len(prefixes)} thuật toán: {prefixes}")
        print(f"               Các mốc N: {ns}")

print("\nDữ liệu đã nạp xong và sẵn sàng để vẽ đồ thị!")

In [ ]:
# ─────────────────────────────────────────────
# 4. HÀM VẼ ĐỒ THỊ THEO ALPHA
# ─────────────────────────────────────────────

def plot_metric_vs_n(ax, alpha_data, n_list, metric_mean, metric_sem,
                     ylabel, title, show_legend=False):
    """
    Vẽ một metric vs N trên ax.
    alpha_data: {prefix -> {n -> metrics}}
    """
    plotted_any = False
    
    # Nếu n_list không được chỉ định, lấy union của tất cả N có sẵn
    if n_list is None:
        target_ns = sorted(set(n for p in alpha_data for n in alpha_data[p].keys()))
    else:
        target_ns = sorted(n_list)

    for idx, prefix in enumerate(sorted(alpha_data.keys())):
        ns, means, sems = [], [], []
        for n in target_ns:
            m = alpha_data[prefix].get(n)
            if m and metric_mean in m:
                ns.append(n)
                means.append(m[metric_mean])
                sems.append(m.get(metric_sem, 0.0))
        if not ns:
            continue
        ns    = np.array(ns)
        means = np.array(means)
        sems  = np.array(sems)
        color  = get_algo_color(prefix, idx)
        label  = get_algo_label(prefix)
        marker = get_algo_marker(prefix)
        ax.plot(ns, means, color=color, label=label, marker=marker, linewidth=2)
        ax.fill_between(ns, means - sems, means + sems, alpha=0.15, color=color)
        plotted_any = True

    ax.set_xscale('log')
    ax.set_xlabel('N (samples)')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    if show_legend and plotted_any:
        ax.legend(loc='best', fontsize=8)


def plot_experiment_all_alphas(exp_name, exp_data, n_list=None, save_dir=None):
    """
    Vẽ toàn bộ đồ thị cho một thực nghiệm (linear, cosine, simglucose, ...).
    Layout: rows = alpha, cols = [CVaR Suboptimality, Regret, Achieved CVaR, (Accuracy nếu có)]
    """
    alphas = sorted(exp_data.keys())
    if not alphas:
        print(f'Không có dữ liệu cho {exp_name}')
        return

    # Xác định các metrics có trong dữ liệu
    sample_m = {}
    for a in alphas:
        for p in exp_data[a]:
            for n in exp_data[a][p]:
                sample_m.update(exp_data[a][p][n])
                break

    METRICS = [
        ('subopt_mean',  'subopt_sem',  'CVaR Suboptimality',   'CVaR Sub-opt ↓'),
        ('regret_mean',  'regret_sem',  'Regret',               'Mean Regret ↓'),
        ('gt_cvar_mean', 'gt_cvar_sem', 'Achieved CVaR',        'Achieved CVaR ↑'),
    ]
    if 'acc_mean' in sample_m:
        METRICS.append(('acc_mean', 'acc_sem', 'Accuracy', 'Accuracy (%) ↑'))

    n_rows = len(alphas)
    n_cols = len(METRICS)
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(5.2 * n_cols, 3.8 * n_rows),
                             squeeze=False)
    fig.suptitle(f'Dataset / Experiment: {exp_name.upper()}', fontsize=16, fontweight='bold', y=1.01)

    for row_idx, alpha in enumerate(alphas):
        alpha_data = exp_data[alpha]
        for col_idx, (m_mean, m_sem, col_title, ylabel) in enumerate(METRICS):
            ax = axes[row_idx][col_idx]
            show_legend = (col_idx == 0)   # chỉ hiện legend ở cột đầu tiên
            title = f'α={alpha}  |  {col_title}'
            plot_metric_vs_n(ax, alpha_data, n_list, m_mean, m_sem,
                             ylabel=ylabel, title=title, show_legend=show_legend)

    plt.tight_layout()
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        fpath = os.path.join(save_dir, f'convergence_{exp_name}.pdf')
        plt.savefig(fpath, bbox_inches='tight')
        print(f'Đã lưu đồ thị: {fpath}')
    plt.show()

print('Hàm vẽ đồ thị đã định nghĩa.')

In [ ]:
# ─────────────────────────────────────────────
# 5. VẼ: TẤT CẢ EXPERIMENT (SYNTHETIC & SIMGLUCOSE)
# Layout: rows = alpha, cols = metric
# ─────────────────────────────────────────────

SAVE_DIR = None   # đặt thành 'figures/' nếu muốn lưu bản PDF

for exp_name in sorted(all_experiment_data.keys()):
    print(f'\n=== ĐỒ THỊ HỘI TỤ: {exp_name.upper()} ===')
    plot_experiment_all_alphas(exp_name, all_experiment_data[exp_name], n_list=N_LIST, save_dir=SAVE_DIR)

In [ ]:
# ─────────────────────────────────────────────
# 6. SO SÁNH TÁC ĐỘNG CỦA ALPHA THEO TỪNG THUẬT TOÁN (compact)
# Mỗi figure = 1 experiment, mỗi subplot = 1 thuật toán,
# mỗi line = 1 alpha
# ─────────────────────────────────────────────

ALPHA_COLORS = {
    0.01: '#003f5c',
    0.05: '#58508d',
    0.1:  '#bc5090',
    0.15: '#ff6361',
    0.2:  '#ffa600',
}
ALPHA_LINESTYLES = {
    0.01: '-',
    0.05: '--',
    0.1:  '-.',
    0.15: ':',
    0.2:  (0, (3, 1, 1, 1)),
}


def plot_algo_alpha_comparison(exp_name, exp_data, metric_mean, metric_sem,
                               ylabel, n_list=None, save_dir=None):
    """
    1 figure: mỗi subplot = 1 thuật toán.
    Trong mỗi subplot: mỗi đường biểu diễn 1 alpha.
    """
    alphas = sorted(exp_data.keys())
    prefixes = sorted(set(p for a in alphas for p in exp_data[a].keys()))
    if not prefixes:
        return

    n_cols = min(4, len(prefixes))
    n_rows = (len(prefixes) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(4.8 * n_cols, 3.8 * n_rows),
                             squeeze=False)
    fig.suptitle(f'{exp_name.upper()}  –  {ylabel} across α', fontsize=15, fontweight='bold', y=1.02)

    flat_axes = axes.flatten()

    for idx, prefix in enumerate(prefixes):
        ax = flat_axes[idx]
        for alpha in alphas:
            d_alpha = exp_data[alpha].get(prefix, {})
            target_ns = sorted(n_list) if n_list else sorted(d_alpha.keys())
            ns, means, sems = [], [], []
            for n in target_ns:
                m = d_alpha.get(n)
                if m and metric_mean in m:
                    ns.append(n)
                    means.append(m[metric_mean])
                    sems.append(m.get(metric_sem, 0.0))
            if not ns:
                continue
            ns    = np.array(ns)
            means = np.array(means)
            sems  = np.array(sems)
            c  = ALPHA_COLORS.get(alpha, '#555555')
            ls = ALPHA_LINESTYLES.get(alpha, '-')
            ax.plot(ns, means, color=c, linestyle=ls, label=f'α={alpha}', marker='o', markersize=4)
            ax.fill_between(ns, means - sems, means + sems, alpha=0.12, color=c)

        ax.set_xscale('log')
        ax.set_xlabel('N')
        ax.set_ylabel(ylabel)
        ax.set_title(get_algo_label(prefix), fontsize=11)
        ax.legend(fontsize=8)

    # Xoá các subplot trống
    for j in range(idx + 1, len(flat_axes)):
        fig.delaxes(flat_axes[j])

    plt.tight_layout()
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        safe_metric = metric_mean.replace('_mean', '')
        fpath = os.path.join(save_dir, f'alpha_compare_{exp_name}_{safe_metric}.pdf')
        plt.savefig(fpath, bbox_inches='tight')
        print(f'Đã lưu: {fpath}')
    plt.show()


for exp_name in sorted(all_experiment_data.keys()):
    print(f'\n=== {exp_name.upper()}: CVaR Suboptimality vs Alpha ===')
    plot_algo_alpha_comparison(exp_name, all_experiment_data[exp_name],
                               'subopt_mean', 'subopt_sem',
                               'CVaR Suboptimality ↓', n_list=N_LIST, save_dir=SAVE_DIR)
    print(f'=== {exp_name.upper()}: Regret vs Alpha ===')
    plot_algo_alpha_comparison(exp_name, all_experiment_data[exp_name],
                               'regret_mean', 'regret_sem',
                               'Regret ↓', n_list=N_LIST, save_dir=SAVE_DIR)

In [ ]:
# ─────────────────────────────────────────────
# 7. BẢNG TỔNG HỢP: Suboptimality & Regret tại N lớn nhất
# ─────────────────────────────────────────────

for exp_name in sorted(all_experiment_data.keys()):
    exp_data = all_experiment_data[exp_name]
    alphas = sorted(exp_data.keys())
    all_algos = sorted(set(p for a in alphas for p in exp_data[a].keys()))
    
    # Tìm N lớn nhất cho experiment này
    all_ns = set(n for a in alphas for p in exp_data[a] for n in exp_data[a][p].keys())
    if not all_ns:
        continue
    n_max = max(all_ns)

    print(f"\n{'='*75}")
    print(f"  BẢNG TỔNG HỢP: {exp_name.upper()} | CVaR Suboptimality @ N = {n_max}")
    print('='*75)
    header = f"  {'Alpha':>8} | " + " | ".join(f"{get_algo_label(p):>24}" for p in all_algos)
    print(header)
    print("  " + "-" * len(header))

    for alpha in alphas:
        row = f"  {alpha:>8} |"
        for p in all_algos:
            m = exp_data[alpha].get(p, {}).get(n_max)
            if m and 'subopt_mean' in m:
                val = f"{m['subopt_mean']:.4f} ± {m['subopt_sem']:.4f}"
            else:
                val = "N/A"
            row += f" {val:>24} |"
        print(row)